<a href="https://colab.research.google.com/github/dsci3d/messyverse/blob/main/notebooks/02_doi-crossref.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab oeffnen"/></a>

# DOIs zu Zitationen anreichern -- ueber Crossref

Ein **DOI** (Digital Object Identifier) ist die eindeutige Kennung einer wissenschaftlichen Publikation. **Crossref** liefert dazu die vollstaendigen Angaben. Dein Auftrag in zwei Teilen: (1) zu jedem DOI aus `katalog/dois.txt` Titel, Zeitschrift und Jahr holen; (2) eine Themen-Suche ueber mehrere Seiten blaettern (Pagination) und die Treffer zu einer Liste fuehren.

In [ ]:
# SETUP (Black Box -- einmal ausfuehren). Holt deine Arbeitskopie des Uebungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/dsci3d/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Fixtures-first

Die Crossref-Antworten liegen in `api-fixtures/crossref/` -- eine Datei pro DOI (der `/` im DOI ist im Dateinamen durch `_` ersetzt) sowie drei Such-Seiten `suche_seite-*.json`. Vertrag: `api-fixtures/crossref/README.md`.

## Schritt 1 -- die Lage ansehen

In [ ]:
print(open("katalog/dois.txt").read())
import json
m = json.load(open("api-fixtures/crossref/10.1038_sdata.2016.18.json"))["message"]
print(m["title"][0], "|", m["container-title"][0], "|", m["issued"]["date-parts"][0][0])

## Schritt 2 -- die KI prompten (Teil 1: DOI -> Zitation)

> *Lies `katalog/dois.txt`. Lade fuer jeden DOI `api-fixtures/crossref/<doi mit _ statt />.json`. Zieh aus `message` den Titel (`title[0]`), die Zeitschrift (`container-title[0]`) und das Jahr (`issued.date-parts[0][0]`). Baue ein dict `ergebnis`: DOI -> {'titel':..., 'container':..., 'jahr':...}.*

In [ ]:
# Code deines KI-Assistenten hier einfuegen.
# Ziel: dict `ergebnis` -> DOI: {'titel':..., 'container':..., 'jahr':...}


## Schritt 3 -- Teil 1 pruefen (Black Box)

In [ ]:
import json
soll = json.load(open("loesungen/doi_lookup.golden.json"))
try:
    ergebnis
except NameError:
    print("Noch kein `ergebnis` (Teil 1).")
else:
    fehlen = [d for d in soll if d not in ergebnis]
    ab = [d for d in soll if d in ergebnis and ergebnis[d].get("titel") != soll[d]["titel"]]
    print("OK -- alle Zitationen stimmen." if not fehlen and not ab
          else f"fehlen={fehlen} titel_ab={ab}")

## Schritt 4 -- die KI prompten (Teil 2: Pagination)

Eine Themen-Suche liefert mehr Treffer, als auf eine Seite passen. Man blaettert ueber `rows` (Treffer pro Seite) und `offset` (Startposition). Die drei Seiten liegen schon gespeichert.

> *Lade die drei Dateien `api-fixtures/crossref/suche_seite-1_offset-0.json`, `...seite-2...`, `...seite-3...`. Sammle aus jeder `message.items` die DOIs zu einer Liste `treffer`.*

In [ ]:
# Code deines KI-Assistenten hier einfuegen.
# Ziel: Liste `treffer` mit allen DOIs der drei Seiten


## Schritt 5 -- Teil 2 pruefen (Black Box)

In [ ]:
import json
soll = json.load(open("loesungen/doi_suche.golden.json"))
try:
    treffer
except NameError:
    print("Noch keine `treffer`-Liste (Teil 2).")
else:
    print(f"OK -- {len(treffer)} Treffer zusammengefuehrt." if len(treffer) == soll["anzahl"]
          else f"Erwartet {soll['anzahl']} Treffer, bekommen {len(treffer)}. Alle drei Seiten erfasst?")

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Pruefzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurueck und bitte um eine korrigierte Fassung. Die haeufigsten Ursachen sind uebersehene Sonderfaelle -- leere Eintraege, fehlende Felder, ungewohnte Schreibweisen.

## Bonus (live)

Optional: lass die KI die Suche gegen den echten Endpoint bauen (`https://api.crossref.org/works?query.bibliographic=...&rows=5&offset=0`) und die Gesamttrefferzahl (`message.total-results`) ausgeben -- so siehst du, warum man blaettert.